In [1]:
from datasets import Dataset
import pandas as pd
import evaluate

In [2]:
# o parte din punctaj se poate lua foarte usor cu t5-base

In [3]:
train = pd.read_csv("train.csv")

In [4]:
train

,SampleID,Problem,Solution
0,0,Solve for x: x*x = (4),x = -2 or x = 2
1,1,Solve for x: (5)*x + (17) = (10),x = -7/5
2,2,(-25) + (-24) + (-23) + (19),The result of (-25) + (-24) + (-23) + (19) is ...
3,3,(4) + (-5) + (-1),The result of (4) + (-5) + (-1) is -2.
4,4,Solve for x: (7)*x + (20) = (46),x = 26/7
...,...,...,...
1495,1495,(11) * (10),The result of (11) * (10) is 110.
1496,1496,Solve for x: (6)*x + (9) = (-4),x = -13/6
1497,1497,Solve for x: x*x = (16),x = -4 or x = 4
1498,1498,(-12) + (21) + (23) + (6) + (22),The result of (-12) + (21) + (23) + (6) + (22)...


In [5]:
from sklearn.model_selection import train_test_split

In [6]:
df,eval = train_test_split(train,test_size=0.2)

In [7]:
df

,SampleID,Problem,Solution
1361,1361,Solve for x: x + (-14) = (-15),x = -1
728,728,Solve for x: (8)*x + (19) = (44),x = 25/8
181,181,Solve for x: x*x = (4),x = -2 or x = 2
180,180,Solve for x: x*x = (1),x = -1 or x = 1
270,270,(-18) + (-1) + (8),The result of (-18) + (-1) + (8) is -11.
...,...,...,...
1405,1405,(17) + (2) + (-27),The result of (17) + (2) + (-27) is -8.
42,42,Solve for x: x + (19) = (-14),x = -33
395,395,Solve for x: x + (20) = (-4),x = -24
65,65,Solve for x: (8)*x + (-3) = (88),x = 91/8


In [8]:
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Trainer,TrainingArguments

In [9]:
accuracy = evaluate.load("accuracy")

In [10]:
import torch 
device = torch.device("cuda")

In [11]:
df = Dataset.from_pandas(df)
eval = Dataset.from_pandas(eval)

In [12]:
model_name = "t5-large" # test model
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [13]:
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

In [14]:
df = df.map(lambda x: tokenizer(x['Problem'],text_target=x['Solution'],truncation=True),batched=True,remove_columns=['Problem','Solution','SampleID'])
eval = eval.map(lambda x: tokenizer(x['Problem'],text_target=x['Solution'],truncation=True),batched=True,remove_columns=['Problem','Solution','SampleID'])

Map:   0%|          | 0/1200 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [15]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model)

In [16]:
import numpy as np

In [17]:
args=TrainingArguments(
    per_device_eval_batch_size=32,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    num_train_epochs=10,
    weight_decay=0.01,
   # fp16=torch.cuda.is_available(),
    logging_steps=15,
)

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
    train_dataset=df,
    eval_dataset=eval,
    

)

In [19]:
trainer.train()

Step,Training Loss
15,3.012301
30,1.627428
45,1.049195
60,0.808450
75,0.755387
90,0.645769
105,0.595366
120,0.625031
135,0.551865
150,0.478792


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=750, training_loss=0.5510580590565999, metrics={'train_runtime': 904.2824, 'train_samples_per_second': 13.27, 'train_steps_per_second': 0.829, 'total_flos': 942878097408000.0, 'train_loss': 0.5510580590565999, 'epoch': 10.0})

In [20]:
test = pd.read_csv("test.csv")

In [21]:
test

,SampleID,Problem
0,0,Solve for x: (15)*x = (58)
1,1,(6) + (22) + (-29)
2,2,Solve for x: (1)*x = (-16)
3,3,Solve for x: x*x = (4)
4,4,Solve for x: x*x = (49)
...,...,...
95,95,(-15) + (1) + (8)
96,96,Solve for x: x*x = (16)
97,97,Solve for x: (2)*x + (6) = (-3)
98,98,Solve for x: x*x = (16)


In [22]:
texts = test['Problem'].to_list()

In [23]:
predicts = []
batch_size = 32

for i in range(0, len(texts), batch_size):
    text_batch = texts[i:i+batch_size]
    
    inputs = tokenizer(text_batch, truncation=True, padding=True, return_tensors="pt").to(device)    
    with torch.no_grad():
        outputs = model.generate(**inputs)
    
    predicts.extend([tokenizer.decode(t, skip_special_tokens=True) for t in outputs])

In [24]:
predicts

['x = 58/15',
 'The result of (6) + (22) + (-29) is -31.',
 'x = -16',
 'x = -2 or x = 2',
 'x = -7 or x = 7',
 'x = 35',
 'The result of (46) * (-12) is -54.',
 'The result of (14) + (-35) is -55.',
 'x = -2 or x = 2',
 'The result of (17) + (11) + (21) is 58.',
 'x = -23',
 'x = -7/10',
 'x = -10',
 'The result of (-15) + (-27) + (-30) + (-8) +',
 'x = 76/8',
 'The result of (-2) + (-2) + (23) + (24) is -45',
 'x = -6 or x = 6',
 'x = -1/4',
 'x = 12',
 'x = -5 or x = 5',
 'x = 59/2',
 'x = -1',
 'The result of (9) - (-34) is -32.',
 'The result of (47) / (6) is 0.25.',
 'x = -33/7',
 'x = -4 or x = 4',
 'The result of (-17) + (4) + (-15) + (22) + (-2)',
 'x = 39/3',
 'The result of (22) + (-27) + (-14) + (-11) is ',
 'x = -23/6',
 'The result of (0) + (5) + (-9) is -32.',
 'x = -63/8',
 'x = -10 or x = 10',
 'The result of (18) + (8) is 18.',
 'x = -4 or x = 4',
 'The result of (2) / (21) is 0.3333333333333333333333333.',
 'The result of (5) / (13) is 0.',
 'x = -81/18',
 'x = -99/1

In [29]:
test['datapointID'] = 1
test['answer'] = predicts
test = test.drop('Problem',axis = 1)

In [30]:
test

,SampleID,datapointID,answer
0,0,1,x = 58/15
1,1,1,The result of (6) + (22) + (-29) is -31.
2,2,1,x = -16
3,3,1,x = -2 or x = 2
4,4,1,x = -7 or x = 7
...,...,...,...
95,95,1,The result of (-15) + (1) + (8) is -23.
96,96,1,x = -4 or x = 4
97,97,1,x = -1
98,98,1,x = -4 or x = 4


In [25]:
groundtruth_file = pd.read_csv("groundtruth.csv")

In [ ]:
test.to_csv("submission_only_For_test.csv")

In [ ]:
# URMATOAREA FUNCTIE ESTE GENERATA CU AJUTORUL LLM PENTRU A PUTEA CALCULA METRICA!!!!:
import pandas as pd

def evaluate_submission(groundtruth_file, submission_file):
    """
    Compara groundtruth cu submission și calculează scorul MAS.
    MAS = 0.6 * NumericScore + 0.3 * ParenthesesScore + 0.1 * StepClarityScore
    """
    # 🔹 Încarcă fișierele
    gt = pd.read_csv(groundtruth_file)
    sub = pd.read_csv(submission_file)

    # 🔹 Verificăm că SampleID și datapointID coincid
    merged = pd.merge(gt, sub, on=['SampleID', 'datapointID'], how='inner', suffixes=('_gt','_sub'))
    
    scores = []

    for _, row in merged.iterrows():
        answer_gt = row['answer_gt'].strip()
        answer_sub = row['answer_sub'].strip()

        # 1️⃣ NumericScore: verificăm dacă rezultatul numeric final e corect
        # Căutăm ultimele numere din string, simplu
        import re

        def extract_numbers(s):
            # extrage toate numerele cu semn
            return [float(n) for n in re.findall(r'-?\d+\.?\d*', s)]

        nums_gt = extract_numbers(answer_gt)
        nums_sub = extract_numbers(answer_sub)
        numeric_score = 1 if nums_gt == nums_sub else 0

        # 2️⃣ ParenthesesScore: dacă toate parantezele din groundtruth sunt prezente în submission
        # simplu: verificăm dacă numărul de '(' și ')' coincide
        parentheses_score = 1 if answer_gt.count('(') == answer_sub.count('(') and \
                               answer_gt.count(')') == answer_sub.count(')') else 0

        # 3️⃣ StepClarityScore: dacă submission conține cuvinte explicite: "The result of" sau "x ="
        clarity_keywords = ["The result of", "x ="]
        step_clarity_score = 1 if any(k in answer_sub for k in clarity_keywords) else 0

        # 🔹 MAS
        MAS = 0.6 * numeric_score + 0.3 * parentheses_score + 0.1 * step_clarity_score
        scores.append(MAS)

    # 🔹 Scor final
    final_score = sum(scores) / len(scores)
    print(f"Numar probleme evaluate: {len(scores)}")
    print(f"MAS final: {final_score:.4f} (din 1.0)")

    return final_score

# 🔹 Exemplu de utilizare
evaluate_submission("groundtruth.csv", "submission_only_For_test.csv")

Numar probleme evaluate: 100
MAS final: 0.5680 (din 1.0)


0.568